# 第14集：处理缺失数据

> 原视频 P14：3.4 pandas 处理丢失数据｜时长：7分24秒

本笔记严格依照原聊天记录中的讲解顺序整理。每个知识点先解释含义，再给出代码和记录中的预期输出；标为旧写法或故意报错的片段只用于阅读，不作为可执行单元。

## 运行前准备

原聊天记录在这里沿用前一集已经导入的库。为了让本 Notebook 能独立运行，先补上相同的导入。


In [ ]:
import numpy as np
import pandas as pd


原视频内容：[Pandas处理缺失数据](https://mofanpy.com/tutorials/data-manipulation/np-pd/pd-nan)

缺失值通常表示：

```text
这个位置没有有效数据
```

常见形式有：

```python
np.nan
pd.NA
None
```

## 1. 创建含NaN的数据


In [ ]:
dates = pd.date_range("20130101", periods=6)

df = pd.DataFrame(
    np.arange(24).reshape((6, 4)),
    index=dates,
    columns=["A", "B", "C", "D"]
)

df.iloc[0, 1] = np.nan
df.iloc[1, 2] = np.nan

print(df)


输出：

```text
             A     B     C   D
2013-01-01   0   NaN   2.0   3
2013-01-02   4   5.0   NaN   7
2013-01-03   8   9.0  10.0  11
2013-01-04  12  13.0  14.0  15
2013-01-05  16  17.0  18.0  19
2013-01-06  20  21.0  22.0  23
```

B、C列出现小数，是因为传统整数列加入 `NaN` 后通常会转换成浮点数。Pandas目前还提供可空整数类型等其他缺失值表示方式。[Pandas缺失数据文档](https://pandas.pydata.org/docs/user_guide/missing_data.html)

---

## 2. `dropna()`删除缺失数据

删除包含缺失值的行：


In [ ]:
result = df.dropna(axis=0, how="any")

print(result)


输出：

```text
             A     B     C   D
2013-01-03   8   9.0  10.0  11
2013-01-04  12  13.0  14.0  15
2013-01-05  16  17.0  18.0  19
2013-01-06  20  21.0  22.0  23
```

参数：

```text
axis=0 → 删除行
axis=1 → 删除列

how="any" → 只要有一个缺失值就删除
how="all" → 必须全部缺失才删除
```

删除含缺失值的列：


In [ ]:
print(df.dropna(axis=1, how="any"))


输出：

```text
             A   D
2013-01-01   0   3
2013-01-02   4   7
2013-01-03   8  11
2013-01-04  12  15
2013-01-05  16  19
2013-01-06  20  23
```

B、C列存在缺失值，所以整列被删除。

---

## 3. `dropna()`默认不会修改原DataFrame


In [ ]:
result = df.dropna()

print(result)
print(df)


`result` 是删除后的新DataFrame，而 `df` 默认仍然保留原数据。

如果想保留处理结果，推荐明确赋值：

```python
df = df.dropna()
```

---

## 4. `fillna()`填充缺失值

用0填充：


In [ ]:
result = df.fillna(0)

print(result)


输出：

```text
             A     B     C   D
2013-01-01   0   0.0   2.0   3
2013-01-02   4   5.0   0.0   7
2013-01-03   8   9.0  10.0  11
2013-01-04  12  13.0  14.0  15
2013-01-05  16  17.0  18.0  19
2013-01-06  20  21.0  22.0  23
```

也可以每列使用不同填充值：


In [ ]:
result = df.fillna({
    "B": -1,
    "C": 999
})

print(result.head(2))


输出：

```text
            A    B      C  D
2013-01-01  0 -1.0    2.0  3
2013-01-02  4  5.0  999.0  7
```

---

## 5. `isna()`检查缺失值


In [ ]:
print(df.isna())


输出：

```text
                A      B      C      D
2013-01-01  False   True  False  False
2013-01-02  False  False   True  False
2013-01-03  False  False  False  False
2013-01-04  False  False  False  False
2013-01-05  False  False  False  False
2013-01-06  False  False  False  False
```

- `True`：这个位置缺失
- `False`：这个位置有数据

视频使用：

```python
df.isnull()
```

`isnull()` 和 `isna()` 效果相同。现在更常看到：

```python
df.isna()
```

---

## 6. 判断整个表格是否存在缺失值


In [ ]:
print(df.isna().any())


输出：

```text
A    False
B     True
C     True
D    False
dtype: bool
```

这是每一列是否含缺失值。

再执行一次 `.any()`：


In [ ]:
print(df.isna().any().any())


输出：

```text
True
```

意思是整个DataFrame中至少存在一个缺失值。

统计每列缺失数量：


In [ ]:
print(df.isna().sum())


输出：

```text
A    0
B    1
C    1
D    0
dtype: int64
```

---
